In [1]:
import sys
from pathlib import Path

In [2]:
PROJECT_ROOT = Path().resolve().parent.parent.parent

In [3]:
# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

In [5]:
import pandas as pd
from src.pipeline.utils.loader import load_jsonl
from src.pipeline.utils.document import Document
from collections import Counter
from typing import Any, Dict, Iterator

In [6]:
DATA_PATH = PROJECT_ROOT / "outputs" / "pipeline_test" / "2_minhash_deduplication"

In [12]:
DATA_PATH = PROJECT_ROOT / "data" / "processed"

In [ ]:
def extract_field_paths(obj: Any, prefix: str = "") -> set[str]:
    """
    Recursively extract all field paths from nested metadata.

    Example:
    {
        "author": {
            "name": "Alice"
        },
        "tags": [{"label": "x"}]
    }

    -> {
        "author",
        "author.name",
        "tags",
        "tags[].label"
    }
    """
    fields = set()

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key

            fields.add(path)

            fields.update(extract_field_paths(value, path))

    elif isinstance(obj, list):
        for item in obj:
            list_prefix = f"{prefix}[]"
            fields.update(extract_field_paths(item, list_prefix))

    return fields


In [66]:
def analyze_documents(doc_iter: Iterator[Document]):
    total_docs = 0
    field_counts = Counter()

    for doc in doc_iter:
        total_docs += 1

        fields_in_doc = extract_field_paths(doc.metadata)

        # Count each field once per document
        field_counts.update(fields_in_doc)

    return total_docs, field_counts

In [7]:
def iterate_directory(directory:Path) -> Iterator[Document]:
    for file in directory.glob('*.jsonl'):
        documents = load_jsonl(str(file), field_map={"id":"id", "text":"text"}, load_metadata=True, metadata_fields="*")
        for doc in documents:
            yield doc

In [73]:
documents = iterate_directory(DATA_PATH)

In [10]:
heuristics_data_path = PROJECT_ROOT / "outputs" / "heuristics_filtering"

In [9]:
import json

In [14]:
with open(str(heuristics_data_path / f"count_sentences_with_javascript.jsonl"), "w") as file:
    for document in documents:
        if document.metadata['count_sentences_with_javascript'] > 0:
                doc_flat = document.to_dict(include_metadata = True, metadata_fields = "*", flatten=True)
                json_record = json.dumps(doc_flat, ensure_ascii=False)
                file.write(json_record + '\n')
        

In [16]:
affected_docs = ["americasnlp2024_27377", "americasnlp2024_27378", "fineweb-2_365", "fineweb-2_748", "fineweb-2_1651", "fineweb-2_3042",
                      "fineweb-2_4016"," fineweb-2_4500", "fineweb-2_7468", "fineweb-2_11951", "fineweb-2_12334", "fineweb-2_13237", "fineweb-2_14628",
                      "fineweb-2_15602", "fineweb-2_16086", "fineweb-2_19054", "fineweb-2_23537", "fineweb-2_23920", "fineweb-2_24823", "fineweb-2_26214",
                      "fineweb-2_27188", "fineweb-2_27672", "fineweb-2_30640", "fineweb-2_36209", "fineweb-2_37078", "fineweb-2_37116", "fineweb-2_37527",
                      "fineweb-2_38176", "fineweb-2_41411", "fineweb-2_43121", "fineweb-2_44199", "fineweb-2_44993"]

In [20]:
rows = []
for document in documents:
    if document.metadata['count_sentences_with_javascript'] > 0:
        row = {
            "id": document.id,
            "text": document.text
        }
        for field, value in document.metadata.items():
            if type(value) != "dict":
                row[field] = value
    
        row["remove"] = document.id in affected_docs

        rows.append(row)

In [60]:
low_lang_score = 0
for document in documents:
    if document.metadata['language_score'] < 0.7:
        low_lang_score +=1

In [61]:
low_lang_score

44606

In [21]:
import pandas as pd

df = pd.DataFrame(rows)

In [43]:
df.columns.tolist()

['id',
 'text',
 'count_sentences_with_low_guarani_proportion',
 'ratio_stopwords_to_non_stopwords',
 'corpus',
 'ratio_symbols_to_words',
 'count_lorem_ipsum_sentences',
 'count_sentences_with_javascript',
 'duplicate',
 'min_sentence_length',
 'avg_character_repetition_ratio_per_sentence',
 'avg_words_per_sentence',
 'num_words_split',
 'source',
 'num_chars',
 'count_sentences_starting_with_bullet',
 'avg_sentence_length',
 'count_sentences_with_curly_bracket',
 'count_sentences_with_legal_phrases',
 'language_identification_method',
 'average_words_in_sentences_starting_with_capital',
 'num_words_punct_spacy',
 'avg_numbers_per_sentence',
 'count_sentences_ending_with_ellipsis',
 'corpus_file',
 'avg_uppercase_letters_per_sentence',
 'language_score_source',
 'avg_alphanumeric_characters_per_sentence',
 'max_sentence_length',
 'avg_characters_per_sentence',
 'url',
 'count_sentences_without_terminal_punctuation',
 'language',
 'language_script',
 'language_score',
 'mean_word_lengt

In [26]:
df['remove'] = df.remove.astype(int)

In [30]:
import plotly.express as px
# # Here we use a column with categorical data
# fig = px.histogram(df, x="day")
# fig.show()

In [37]:
corr = df.corr(numeric_only=True)

In [31]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
)

fig.update_layout(
    title="Correlation Heatmap",
    width=1000,
    height=1000,
)

fig.show()

In [ ]:
fig = px.box(
    df, 
    x="remove",           # Primary category on X-axis
    y="avg_alphanumeric characters_per_c",     # Quantitative values on Y-axis
    color="remove",     # Separate sub-categories by color
    title="Grouped Box Plot with Separate Categories"
)

# Display the plot
fig.show()

In [63]:
fig = px.scatter(
    df,
    x="language_score",
    y="avg_numbers_per_sentence",
    #size="language_score",          # dot size
    color="remove",              # dot colour
    hover_data=["id"],
    size_max=40,
    opacity=0.7,
    title="Low Guarani Proportion vs Javascript Sentences"
)

fig.show()

In [74]:
rows_all = []
for document in documents:
    row = {
        "id": document.id,
        "text": document.text
    }
    for field, value in document.metadata.items():
        if type(value) != "dict":
            row[field] = value

    #row["remove"] = document.id in affected_docs

    rows_all.append(row)

In [75]:
df_all = pd.DataFrame(rows_all)

In [89]:
fig = px.scatter(
    df_all[df_all['ratio_symbols_to_words']>1.27],
    x="language_score",
    y="ratio_symbols_to_words",
    #size="language_score",          # dot size
    #color="remove",              # dot colour
    hover_data=["id"],
    size_max=40,
    opacity=0.7,
    title="Low Guarani Proportion vs Javascript Sentences"
)
fig.show()

In [90]:
df_all.dtypes

id                                                      str
text                                                    str
count_sentences_with_low_guarani_proportion           int64
ratio_stopwords_to_non_stopwords                    float64
corpus                                                  str
ratio_symbols_to_words                              float64
count_lorem_ipsum_sentences                           int64
count_sentences_with_javascript                       int64
duplicate                                            object
min_sentence_length                                   int64
avg_character_repetition_ratio_per_sentence         float64
avg_words_per_sentence                              float64
num_words_split                                       int64
source                                                  str
num_chars                                             int64
count_sentences_starting_with_bullet                  int64
avg_sentence_length                     

In [ ]:
to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'symbols_to_words.jsonl'), orient='records', lines=True)

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [101]:
df_all[df_all['num_words_punct_spacy']<2].to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'n_words_l_2.jsonl'), orient='records', lines=True, force_ascii=False)

In [102]:
df_all[df_all['num_words_punct_spacy']<2]

,id,text,count_sentences_with_low_guarani_proportion,ratio_stopwords_to_non_stopwords,corpus,ratio_symbols_to_words,count_lorem_ipsum_sentences,count_sentences_with_javascript,duplicate,min_sentence_length,...,avg_characters_per_sentence,url,count_sentences_without_terminal_punctuation,language,language_script,language_score,mean_word_length,num_words_no_punct_spacy,avg_word_repetition_ratio_per_sentence,count_bad_words_occurrences
297,joff+_297,tavycho,1,0.0,joff+,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,7.0,unknown,1,grn,Latn,0.0,7.0,1,0.0,1
381,joff+_381,rehe,1,0.0,joff+,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,4.0,unknown,1,grn,Latn,0.0,4.0,1,0.0,0
517,joff+_517,ivai,1,0.0,joff+,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,4.0,unknown,1,grn,Latn,0.0,4.0,1,0.0,0
676,joff+_676,vyro,1,0.0,joff+,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,4.0,unknown,1,grn,Latn,0.0,4.0,1,0.0,1
1306,joff+_1306,heko,1,0.0,joff+,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,4.0,unknown,1,grn,Latn,0.0,4.0,1,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1312431,joemo_1259,chugui,1,0.0,joemo,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,6.0,unknown,1,grn,Latn,0.0,6.0,1,0.0,0
1312471,joemo_1299,tova,1,0.0,joemo,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,4.0,unknown,1,grn,Latn,0.0,4.0,1,0.0,0
1312534,joemo_1362,ipora,1,0.0,joemo,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,5.0,unknown,1,grn,Latn,0.0,5.0,1,0.0,0
1312597,joemo_1425,rehe,1,0.0,joemo,0.0,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,4.0,unknown,1,grn,Latn,0.0,4.0,1,0.0,0


In [ ]:
bar_chart = px.bar(df_all, x='language_score_source')

In [97]:
box_plot = px.box(df, y="num_words_punct_spacy", points="outliers", hover_data=["id"])
#box_plot.write_image(str(plots_dir / f"{metric}_box.png"))

In [98]:
box_plot.show()

In [13]:
diction = {'mean_word_length': 1313005,
 'count_sentences_with_low_guarani_proportion': 1313005,
 'avg_sentence_length': 1313005,
 'avg_uppercase_letters_per_sentence': 1313005,
 'count_sentences_ending_with_ellipsis': 1313005,
 'average_words_in_sentences_starting_with_capital': 1313005,
 'language_score': 1313005,
 'avg_word_repetition_ratio_per_sentence': 1313005,
 'count_sentences_starting_with_bullet': 1313005,
 'count_sentences_with_legal_phrases': 1313005,
 'avg_numbers_per_sentence': 1313005,
 'avg_character_repetition_ratio_per_sentence': 1313005,
 'num_words_no_punct_spacy': 1313005,
 'ratio_symbols_to_words': 1313005,
 'count_lorem_ipsum_sentences': 1313005,
 'num_words_punct_spacy': 1313005,
 'avg_alphanumeric_characters_per_sentence': 1313005,
 'num_words_split': 1313005,
 'avg_words_per_sentence': 1313005,
 'min_sentence_length': 1313005,
 'count_sentences_with_curly_bracket': 1313005,
 'count_sentences_with_javascript': 1313005,
 'ratio_stopwords_to_non_stopwords': 1313005,
 'max_sentence_length': 1313005,
 'count_bad_words_occurrences': 1313005,
 'avg_characters_per_sentence': 1313005,
 'count_sentences_without_terminal_punctuation': 1313005,
 'num_chars': 1313005,
}

In [65]:
documents = iterate_directory(DATA_PATH)

In [28]:
lists_to_explore = ["count_sentences_with_low_guarani_proportion", "count_sentences_with_javascript", "language_score", "avg_alphanumeric_characters_per_sentence", "average_words_in_sentences_starting_with_capital", "count_sentences_with_legal_phrases", "count_sentences_with_javascript"]

In [29]:
rows = []
for doc in documents:
    row = {
        "doc_id":doc.id,
        "text": doc.text
    }

    for field in lists_to_explore:
        row[field] = doc.metadata[field]
    
    rows.append(row)

In [19]:
plots_dir = PROJECT_ROOT/"outputs"/"heuristics_filtering"/"plots"
lists_dir = PROJECT_ROOT/"outputs"/"heuristics_filtering"/"lists"

In [18]:
summary = df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
summary["n_unique"] = df.nunique()

In [19]:
summary

,count,mean,std,min,5%,25%,50%,75%,95%,max,n_unique
mean_word_length,1313005.0,6.143205,1.705869,0.0,4.000000,5.176471,6.000000,6.866667,9.000000,4.200000e+01,49396
count_sentences_with_low_guarani_proportion,1313005.0,2.465986,49.460806,0.0,0.000000,1.000000,1.000000,1.000000,7.000000,4.030900e+04,423
avg_sentence_length,1313005.0,10.747074,15.293038,0.0,1.666667,4.000000,7.000000,13.181818,28.000000,2.309000e+03,15463
avg_uppercase_letters_per_sentence,1313005.0,3.722319,11.204526,0.0,0.500000,1.000000,1.000000,3.000000,11.285714,1.549000e+03,12198
count_sentences_ending_with_ellipsis,1313005.0,0.007373,0.172874,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2.200000e+01,23
average_words_in_sentences_starting_with_capital,1313005.0,0.000333,0.127473,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,8.100000e+01,8
language_score,1313005.0,0.966185,0.134523,0.0,0.807793,0.997044,0.999737,0.999987,1.000010,1.000010e+00,212694
avg_word_repetition_ratio_per_sentence,1313005.0,0.041513,0.078447,0.0,0.000000,0.000000,0.000000,0.063630,0.200000,9.756098e-01,97897
count_sentences_starting_with_bullet,1313005.0,0.054417,2.306242,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,8.300000e+02,120
count_sentences_with_legal_phrases,1313005.0,0.001906,0.044578,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000e+00,7


In [21]:
import plotly.express as px

In [22]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
li_fig = px.box(df, y="count_lorem_ipsum_sentences", points="outliers", hover_data=["doc_id"])
li_fig.show()

In [ ]:
lp_fig = px.box(df, y="count_sentences_with_legal_phrases", points="outliers", hover_data=["doc_id"])
lp_fig.show()

In [26]:
lp_fig.write_html(str(PROJECT_ROOT/"outputs"/"test.html"))

In [82]:
def find_outliers_iqr(df, field):
    q1 = df[field].quantile(0.25)
    q3 = df[field].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[
        (df[field] < lower) |
        (df[field] > upper)
    ]

    return outliers

In [21]:
heu_fil = PROJECT_ROOT/"outputs"/"heuristics_filtering"

In [22]:
df.head()

,doc_id,text,count_sentences_with_low_guarani_proportion,count_sentences_with_javascript
0,joff+_0,"@MICHIKATZE1 🤣🤣🤣🤣🤣 umia kuera, sapy'aitepe👌",0,0
1,joff+_1,Che membyséma avei 🥰,1,0
2,joff+_2,Apostala mante ovy'a hina,1,0
3,joff+_3,@josema1975 Oĩ porãta pya'e. 💪🏾,2,0
4,joff+_4,@Pollo2895 Ko.agaite peve,1,0


In [83]:
outliers_df = find_outliers_iqr(df_all, 'ratio_symbols_to_words')

In [94]:
outliers_df.to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'symbols_to_words.jsonl'), orient='records', lines=True, force_ascii=False)

In [84]:
outliers_df

,id,text,count_sentences_with_low_guarani_proportion,ratio_stopwords_to_non_stopwords,corpus,ratio_symbols_to_words,count_lorem_ipsum_sentences,count_sentences_with_javascript,duplicate,min_sentence_length,...,avg_characters_per_sentence,url,count_sentences_without_terminal_punctuation,language,language_script,language_score,mean_word_length,num_words_no_punct_spacy,avg_word_repetition_ratio_per_sentence,count_bad_words_occurrences
0,joff+_0,"@MICHIKATZE1 🤣🤣🤣🤣🤣 umia kuera, sapy'aitepe👌",0,1.000000,joff+,4.500000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",2,...,43.0,unknown,1,grn,Latn,0.979258,4.500000,10,0.0,0
3,joff+_3,@josema1975 Oĩ porãta pya'e. 💪🏾,2,0.000000,joff+,2.500000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",0,...,15.0,unknown,1,grn,Latn,0.999342,4.000000,6,0.0,0
4,joff+_4,@Pollo2895 Ko.agaite peve,1,0.000000,joff+,2.000000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,25.0,unknown,1,grn,Latn,0.000000,4.000000,3,0.0,0
8,joff+_8,@soyMaleok Nde japu jepe ra'e!! 😂😂😂,2,0.500000,joff+,2.333333,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",0,...,17.0,unknown,1,grn,Latn,0.989687,3.666667,8,0.0,0
10,joff+_10,reko ☹️☹️☹️,1,0.000000,joff+,6.000000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",1,...,11.0,unknown,1,grn,Latn,0.000000,4.000000,7,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1312727,joemo_1555,"@ara_duca Ko arapokõindy, ñandekuerái peve ja'...",2,0.250000,joemo,2.200000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",0,...,31.0,unknown,1,grn,Latn,0.997892,6.600000,11,0.0,0
1312728,joemo_1556,@gabimgaona_ Vy'apave ndeve guara tapehasa por...,1,0.333333,joemo,1.750000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",4,...,63.0,unknown,1,grn,Latn,0.984218,6.250000,10,0.0,0
1312731,joemo_1559,@mariliapatinho Paciencia añete orekoarã 👀🤣🤣🤣❤️❤️,1,0.000000,joemo,3.000000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",3,...,49.0,unknown,1,grn,Latn,0.991740,7.333333,12,0.0,0
1312736,joemo_1564,@ivan_almadaa @pedroliuzzii ojapo pretemporada,1,0.000000,joemo,1.500000,0,0,"{'url': {'has_duplicate': False, 'docs_ids': []}}",2,...,46.0,unknown,1,grn,Latn,0.000000,8.500000,4,0.0,0


In [31]:
for metric in lists_to_explore:
    outliers_df = find_outliers_iqr(df, metric)
    outliers_dict = outliers_df.to_dict(orient="records")

    with open(str(heu_fil / f"{metric}_texts.jsonl"), "w") as file:
        for entry in outliers_dict:
            json_record = json.dumps(entry, ensure_ascii=False)
            file.write(json_record + '\n')

In [32]:
for metric in list(diction.keys()):
    #Generate a box plot
    box_plot = px.box(df, y=metric, points="outliers", hover_data=["doc_id"])
    box_plot.write_image(str(plots_dir / f"{metric}_box.png"))

    #Get the outliers
    outliers_df = find_outliers_iqr(df, metric)
    outliers_list = outliers_df["doc_id"].tolist()

    with open(str(lists_dir / f"{metric}.txt"), "w") as file:
        file.write("\n".join(outliers_list))

    #Generate a histogram
    histogram = px.histogram(df, x=metric, hover_data=["doc_id"], marginal="box")
    histogram.write_image(str(plots_dir / f"{metric}_histogram.png"))

In [29]:
import kaleido

kaleido.get_chrome_sync()

PosixPath('/home/abaez/.local/share/choreographer/deps/chrome-linux64/chrome')

In [ ]:
# fig = px.histogram(
#     df,
#     x="mean_word_length",
#     hover_data=["doc_id"]
# )

In [ ]:
# fig.show()

In [33]:
df.dtypes

id                                                      str
text                                                    str
count_sentences_with_low_guarani_proportion           int64
ratio_stopwords_to_non_stopwords                    float64
corpus                                                  str
ratio_symbols_to_words                              float64
count_lorem_ipsum_sentences                           int64
count_sentences_with_javascript                       int64
duplicate                                            object
min_sentence_length                                   int64
avg_character_repetition_ratio_per_sentence         float64
avg_words_per_sentence                              float64
num_words_split                                       int64
source                                                  str
num_chars                                             int64
count_sentences_starting_with_bullet                  int64
avg_sentence_length                     

In [ ]:
documents

In [34]:
w_js = [
    "opus-all-en_363609", 
    "opus-all-en_363610", 
    "opus-all-en_363611", 
    "opus-all-en_363612", 
    "opus-all-en_381869", 
    "americasnlp2024_27377",
    "americasnlp2024_27378", 
    "fineweb-2_365", 
    "fineweb-2_748", 
    "fineweb-2_1651", 
    "fineweb-2_3042", 
    "fineweb-2_3042",  
    "fineweb-2_4016", 
    "fineweb-2_4500",
    "fineweb-2_7468",
    "fineweb-2_11951",
    "fineweb-2_12334"
    "fineweb-2_13237",
    "fineweb-2_23920",
    "fineweb-2_24823",
    "fineweb-2_26214",
    "fineweb-2_27188", 
    "fineweb-2_27672", 
    "fineweb-2_30640", 
    "fineweb-2_36209", 
    "fineweb-2_37078", 
    "fineweb-2_37116", 
    "fineweb-2_37525", 
    "fineweb-2_37527", 
    "fineweb-2_37952", 
    "fineweb-2_38176", 
    "fineweb-2_38881", 
    "fineweb-2_41411", 
    "fineweb-2_41689", 
    "fineweb-2_41856", 
    "fineweb-2_43121", 
    "fineweb-2_43954", 
    "fineweb-2_44035", 
    "fineweb-2_44199", 
    "fineweb-2_44309", 
    "fineweb-2_44993"
]

In [52]:
len(w_js)

40

In [35]:
metrics = ["count_sentences_with_low_guarani_proportion", "count_sentences_with_javascript", "language_score", "language", "num_chars"]

In [38]:
exp_rows = []
for doc in documents:
    if doc.id in w_js:
        row = {
            "doc_id":doc.id,
            "text": doc.text
        }

        for field in metrics:
            row[field] = doc.metadata[field]
        
        exp_rows.append(row)

In [39]:
exp_df = pd.DataFrame(exp_rows)

In [51]:
import pandas as pd
import plotly.express as px

# Keep only needed columns and drop missing values
plot_df = exp_df.dropna()

fig = px.scatter(
    plot_df,
    x="language_score",
    y="count_sentences_with_low_guarani_proportion",
    #size="language_score",          # dot size
    color="language",              # dot colour
    hover_data=["doc_id", "num_chars", "language_score", "count_sentences_with_javascript"],
    size_max=40,
    opacity=0.7,
    title="Low Guarani Proportion vs Javascript Sentences"
)

# fig.update_layout(
#     xaxis_title="Count Sentences with Low Guarani Proportion",
#     yaxis_title="Count Sentences with Javascript",
#     template="plotly_white"
# )

fig.show()